## Nhiệm vụ 2.1 Mã hóa khóa công khai RSA
Notebook độc lập chỉ thực hiện **Task 2.1**, gồm sinh khóa, hai chiều RSA với số nguyên,
mã hóa chuỗi ra Base64 và giải bốn bản mã đề bài. Không triển khai Task 2.2, 2.3, 2.4.
Chỉ dùng thư viện chuẩn Python 3.10 trở lên; **Run All** không chờ nhập và không cần mạng.

## Yêu cầu và cơ sở toán học
Với hai số nguyên tố phân biệt $p,q$: $n=pq$, $\varphi(n)=(p-1)(q-1)$.
Chọn $1<e<\varphi(n)$ và $\gcd(e,\varphi(n))=1$; tính $d=e^{-1}\pmod{\varphi(n)}$
bằng Euclid mở rộng. Khóa công khai $PU=(e,n)$, khóa riêng $PR=(d,n)$.

- Bảo mật: $C=M^e\bmod n$, khôi phục $M=C^d\bmod n$.
- Chiều xác thực theo đề: $S=M^d\bmod n$, kiểm tra $M'=S^e\bmod n$.
- Miền dữ liệu: $0\le M,C<n$; không tự lấy modulo để che giấu dữ liệu ngoài miền.

Đây là **RSA thuần túy dùng minh họa**. Phép lũy thừa bằng khóa riêng trên cả thông điệp
chưa tạo thành chữ ký số an toàn; cơ chế chữ ký thực tế cần mã hóa băm và padding như RSA-PSS.
Base64 chỉ biểu diễn byte, không tăng tính bảo mật [2].

### Sinh khóa khi thiếu tham số
Số nguyên Python có độ dài tùy ý nên không tồn tại cặp khóa “lớn nhất”. Chương trình cho chọn
`bits` (mặc định **2048 bit**), tạo các tham số còn thiếu bằng `secrets`, kiểm tra nguyên tố
Miller–Rabin 64 vòng; đây là kiểm tra nguyên tố xác suất, không phải chứng minh nguyên tố.
Khi cả p và q đều thiếu, chương trình tạo n có đúng `bits` bit. Nếu cung cấp một số nguyên tố,
chương trình tạo số còn lại có kích thước phù hợp; nếu cung cấp e thì sinh p/q tương thích.
Tham số đã cung cấp mà không hợp lệ sẽ bị từ chối [3, 4].

In [1]:
import base64
import binascii
import math
import secrets
from dataclasses import dataclass


def require_int(value, name, minimum=0):
    if type(value) is not int or value < minimum:
        raise ValueError(f'{name} phải là số nguyên >= {minimum}.')


def inverse_mod(a, modulus):
    """Nghịch đảo modulo bằng Euclid mở rộng."""
    require_int(a, 'a', 1)
    require_int(modulus, 'modulus', 2)
    r0, r1, t0, t1 = modulus, a % modulus, 0, 1
    while r1:
        quotient = r0 // r1
        r0, r1 = r1, r0 - quotient * r1
        t0, t1 = t1, t0 - quotient * t1
    if r0 != 1:
        raise ValueError('Không tồn tại nghịch đảo modulo.')
    return t0 % modulus


def is_probable_prime(n, rounds=64):
    require_int(rounds, 'rounds', 1)
    if type(n) is not int or n < 2:
        return False
    for p in (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37):
        if n % p == 0:
            return n == p
    d, s = n - 1, 0
    while d % 2 == 0:
        d //= 2
        s += 1
    for _ in range(rounds):
        a = secrets.randbelow(n - 3) + 2
        x = pow(a, d, n)
        if x in (1, n - 1):
            continue
        for _ in range(s - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                break
        else:
            return False
    return True


def random_prime(bits, e=None, different_from=None):
    require_int(bits, 'bits của số nguyên tố', 8)
    while True:
        candidate = secrets.randbits(bits) | (1 << (bits - 1)) | 1
        if candidate == different_from:
            continue
        if e is not None and math.gcd(e, candidate - 1) != 1:
            continue
        if is_probable_prime(candidate):
            return candidate


@dataclass(frozen=True)
class RSAKey:
    p: int
    q: int
    n: int
    phi: int
    e: int
    d: int

    @property
    def public(self):
        return self.e, self.n

    @property
    def private(self):
        return self.d, self.n


def generate_key(p=None, q=None, e=None, bits=2048):
    require_int(bits, 'bits', 16)
    missing_p, missing_q = p is None, q is None
    for name, value in [('p', p), ('q', q)]:
        if value is not None and not is_probable_prime(value):
            raise ValueError(f'{name} phải là số nguyên tố.')
    if e is not None:
        require_int(e, 'e', 3)
        if e % 2 == 0:
            raise ValueError('e phải lẻ với các khóa RSA được xét.')
        # Ngăn vòng sinh vô hạn khi e mâu thuẫn với p/q đã cho.
        for value in (p, q):
            if value is not None and math.gcd(e, value - 1) != 1:
                raise ValueError('e không tương thích với p hoặc q đã cho.')
    if missing_p:
        p = random_prime(max(8, bits - q.bit_length()) if q else bits // 2, e)
    if missing_q:
        q = random_prime(max(8, bits - p.bit_length()), e, p)
        if missing_p:
            while (p * q).bit_length() != bits:
                q = random_prime(bits - bits // 2, e, p)
    if p == q:
        raise ValueError('p và q phải khác nhau.')
    n, phi = p * q, (p - 1) * (q - 1)
    if e is None:
        while True:
            e = secrets.randbelow(phi - 3) + 3
            if e < phi and math.gcd(e, phi) == 1:
                break
    if not 1 < e < phi or math.gcd(e, phi) != 1:
        raise ValueError('Cần 1 < e < phi và gcd(e, phi) = 1.')
    return RSAKey(p, q, n, phi, e, inverse_mod(e, phi))


def rsa_apply(value, exponent, n):
    require_int(n, 'n', 3)
    require_int(exponent, 'số mũ', 1)
    require_int(value, 'khối dữ liệu')
    if value >= n:
        raise ValueError('Khối dữ liệu phải nhỏ hơn n.')
    return pow(value, exponent, n)

## Phần 1 Ba cặp khóa theo đề
p₁, q₁, e₁ và p₂, q₂, e₂ đọc ở hệ thập phân; bộ thứ ba đọc ở hệ thập lục phân.
Tính d theo φ(n), đúng công thức đề. In toàn bộ PU, PR và kiểm tra e·d mod φ(n) = 1.

In [2]:
PARAMETERS = [
    (11, 17, 7),
    (20079993872842322116151219, 676717145751736242170789, 17),
    (int('F7E75FDC469067FFDC4E847C51F452DF', 16),
     int('E85CED54AF57E53E092113E62F436F4F', 16), int('0D88C3', 16)),
]
KEYS = [generate_key(p, q, e) for p, q, e in PARAMETERS]
for i, key in enumerate(KEYS, 1):
    print(f'Khóa {i}: n có {key.n.bit_length()} bit')
    print('n =', key.n)
    print('phi =', key.phi)
    print('PU =', key.public)
    print('PR =', key.private)
    print('e*d mod phi =', key.e * key.d % key.phi)
    print()

Khóa 1: n có 8 bit
n = 187
phi = 160
PU = (7, 187)
PR = (23, 187)
e*d mod phi = 1

Khóa 2: n có 164 bit
n = 13588476140342208394395166469647627226674348541791
phi = 13588476140342208394395145712936608632615990219784
PU = (17, 13588476140342208394395166469647627226674348541791)
PR = (7993221259024828467291262184080358019185876599873, 13588476140342208394395166469647627226674348541791)
e*d mod phi = 1

Khóa 3: n có 256 bit
n = 101776877529005912638346811918779931246783058062684819617574643018368103302097
phi = 101776877529005912638346811918779931246144673982897083685478381401683682492836
PU = (886979, 101776877529005912638346811918779931246783058062684819617574643018368103302097)
PR = (24212225287904763939160097464943268930139828978795606022583874367720623008491, 101776877529005912638346811918779931246783058062684819617574643018368103302097)
e*d mod phi = 1



## Phần 2 Hai chiều RSA với M bằng 5
Khóa 1: n = 187, φ(n) = 160, e = 7, d = 23. Cả hai phép khôi phục phải trả về 5.
Chiều khóa riêng chỉ là minh họa toán học cho cách tiếp cận xác thực của đề.

In [3]:
key1 = KEYS[0]
M = 5
C = rsa_apply(M, key1.e, key1.n)
recovered = rsa_apply(C, key1.d, key1.n)
S = rsa_apply(M, key1.d, key1.n)
verified = rsa_apply(S, key1.e, key1.n)
print(f'Bảo mật: 5^7 mod 187 = {C}; {C}^23 mod 187 = {recovered}')
print(f'Xác thực: 5^23 mod 187 = {S}; {S}^7 mod 187 = {verified}')
assert (C, recovered, S, verified) == (146, 5, 180, 5)

Bảo mật: 5^7 mod 187 = 146; 146^23 mod 187 = 5
Xác thực: 5^23 mod 187 = 180; 180^7 mod 187 = 5


## Phần 3 Quy ước chuỗi và Base64
Chuyển chuỗi sang **UTF-8**. Đặt `k = ceil(bit_length(n)/8)` là số byte của mỗi khối bản mã,
`b = max(1, floor((bit_length(n)-1)/8))` là số byte của khối bản rõ.
Với ba khóa: `(b, k) = (1, 1), (20, 21), (31, 32)`.

Dùng **little-endian** cho cả byte bản rõ và bản mã để tương thích với mẫu đề [1].
Mỗi khối mã hóa bằng `pow(m, e, n)`, ghi đúng k byte, nối lại rồi chuyển Base64.
Với khóa 1 có n = 187, chỉ nhận byte < 187; chuỗi yêu cầu là ASCII nên hợp lệ.
Chuỗi tiếng Việt dùng khóa 2, 3 hoặc khóa sinh mới. Đây là quy ước byte của chương trình;
không phải bảng mã hai chữ số minh họa trong hình lý thuyết của đề.

Gói dữ liệu giữ **độ dài UTF-8 gốc**, kích thước khối và thứ tự byte để khôi phục chính xác
khối cuối và byte 0. Không dùng `rstrip(b'\x00')` vì sẽ làm mất dữ liệu thật.
Độ dài và quy ước khối là siêu dữ liệu để giải mã, không phải padding an toàn.

In [4]:
def block_sizes(n):
    require_int(n, 'n', 3)
    return max(1, (n.bit_length() - 1) // 8), (n.bit_length() + 7) // 8


def encrypt_text(text, key):
    if not isinstance(text, str):
        raise ValueError('Thông điệp phải là chuỗi.')
    data = text.encode('utf-8')
    b, k = block_sizes(key.n)
    ciphertext = bytearray()
    for offset in range(0, len(data), b):
        m = int.from_bytes(data[offset:offset + b], 'little')
        c = rsa_apply(m, key.e, key.n)
        ciphertext.extend(c.to_bytes(k, 'little'))
    return {'base64': base64.b64encode(ciphertext).decode('ascii'),
            'length': len(data), 'block_size': b,
            'cipher_width': k, 'byteorder': 'little'}


def decrypt_text(packet, key):
    b, k = block_sizes(key.n)
    length = packet['length']
    require_int(length, 'độ dài')
    if (packet['block_size'], packet['cipher_width'], packet['byteorder']) != (b, k, 'little'):
        raise ValueError('Quy ước gói dữ liệu không khớp khóa.')
    try:
        raw = base64.b64decode(packet['base64'], validate=True)
    except (ValueError, binascii.Error) as error:
        raise ValueError('Base64 không hợp lệ.') from error
    if len(raw) != ((length + b - 1) // b) * k:
        raise ValueError('Độ dài bản mã không khớp siêu dữ liệu.')
    plaintext = bytearray()
    for offset in range(0, len(raw), k):
        c = int.from_bytes(raw[offset:offset + k], 'little')
        m = rsa_apply(c, key.d, key.n)
        width = min(b, length - len(plaintext))
        if m >= 256 ** width:
            raise ValueError('Khối giải mã không khớp kích thước bản rõ.')
        plaintext.extend(m.to_bytes(width, 'little'))
    return plaintext.decode('utf-8')

In [5]:
MESSAGE = 'The University of Information Technology'
PACKETS = [encrypt_text(MESSAGE, key) for key in KEYS]
for i, (key, packet) in enumerate(zip(KEYS, PACKETS), 1):
    print(f'Khóa {i}: {packet["length"]} byte; '
          f'khối rõ {packet["block_size"]} byte; khối mã {packet["cipher_width"]} byte')
    print('Base64:', packet['base64'])
    print('Giải mã:', decrypt_text(packet, key))
    print()
    assert decrypt_text(packet, key) == MESSAGE

Khóa 1: 40 byte; khối rõ 1 byte; khối mã 1 byte
Base64: MrNUTERCYGVUfp1gSk1Mm3dMPUJ3m36DXEpgm0JMMlSws0KbMJuJTQ==
Giải mã: The University of Information Technology

Khóa 2: 40 byte; khối rõ 20 byte; khối mã 21 byte
Base64: AWcKXgyeSmY1g3ZElW7IsbcQSYADnjA2y6yY8Hu9900ymPVaONVGLa4A
Giải mã: The University of Information Technology

Khóa 3: 40 byte; khối rõ 31 byte; khối mã 32 byte
Base64: WaG19PwIkNUlVfdbuta2gEk0MtlxY+ZAhRmTc3sogdGEwROPCkKp9/Sb0MjUUGwmB4F4s+zOU9zhzEy3+mMNXA==
Giải mã: The University of Information Technology



## Phần 4 Giải bốn bản mã đã cho
Loại bỏ xuống dòng trình bày trong Base64, Hex và nhị phân. Với chuỗi nhị phân, mỗi nhóm
8 bit biểu diễn một byte **theo thứ tự xuất hiện**, rồi đọc số nguyên little-endian từ các byte đó.
Không đảo cả chuỗi bit hoặc hiểu chuỗi bit như một số nguyên big-endian để tính RSA.

Hàm dò thử ba khóa, cả big/little-endian, với khối mã k byte (khối cuối có thể thiếu byte 0 ở cuối).
Giải mã rồi lọc các byte ASCII in được; đây là **tiêu chí chọn ứng viên** cho bộ mẫu tiếng Anh,
không phải chứng minh xác thực hay bộ phát hiện khóa tổng quát. Khi có nhiều ứng viên, giữ tất cả.
Mỗi ứng viên được mã hóa lại và so sánh **từng số nguyên bản mã**; phép kiểm tra này không tự
xác định được khóa vì RSA là phép hoán vị trên miền hợp lệ. Kết quả mẫu được kiểm thử bằng bản rõ kỳ vọng.

In [6]:
CIPHERTEXTS = [
    ('Base64', 'raUcesUlOkx/8ZhgodMoo0Uu18sC20yXlQFevSu7W/FDxIy0YRHMyX'
     'cHdD9PBvIT2aUft5fCQEGomiVVPv4I'),
    ('Hex', 'C87F570FC4F699CEC24020C6F54221ABAB2CE0C3'),
    ('Base64', 'Z2BUSkJcg0w4XEpgm0JcMExEQmBlVH6dYEpNTHpMHptMQ7NgT'
     'HlgQrNMQ2BKTQ=='),
    ('Nhị phân', '001010000001010011111111101101110010111011001010111011000110011'
     '110111111001111110110100011001111001100001001010001010100111101'
     '010100110011101110111011110101101100000100'),
]


def decode_source(kind, text):
    compact = ''.join(text.split())
    if kind == 'Base64':
        return base64.b64decode(compact, validate=True)
    if kind == 'Hex':
        return bytes.fromhex(compact)
    if kind == 'Nhị phân':
        if not compact or len(compact) % 8 or set(compact) - {'0', '1'}:
            raise ValueError('Nhị phân phải gồm đủ các nhóm 8 bit.')
        return bytes(int(compact[i:i + 8], 2) for i in range(0, len(compact), 8))
    raise ValueError('Định dạng không hỗ trợ.')


def find_candidates(raw, keys):
    candidates = []
    if not raw:
        return candidates
    for index, key in enumerate(keys, 1):
        b, k = block_sizes(key.n)
        parts = [raw[i:i + k] for i in range(0, len(raw), k)]
        for byteorder in ('little', 'big'):
            values = [int.from_bytes(part, byteorder) for part in parts]
            if any(c >= key.n for c in values):
                continue
            decoded = [rsa_apply(c, key.d, key.n) for c in values]
            chunks = [m.to_bytes(max(1, (m.bit_length() + 7) // 8), byteorder)
                      for m in decoded]
            if any(len(chunk) > b for chunk in chunks):
                continue
            joined = b''.join(chunks)
            if not joined or not all(32 <= byte <= 126 for byte in joined):
                continue
            if not all(rsa_apply(m, key.e, key.n) == c for m, c in zip(decoded, values)):
                continue
            item = {'key': index, 'byteorder': byteorder,
                    'plaintext': joined.decode('ascii'), 'cipher_width': k,
                    'bytes': len(raw), 'blocks': len(parts),
                    'plain_lengths': list(map(len, chunks))}
            # k=1: hai thứ tự byte giống nhau; chỉ lưu một ứng viên.
            if not any(x['key'] == index and x['plaintext'] == item['plaintext'] for x in candidates):
                candidates.append(item)
    return candidates


SOLUTIONS = []
for i, (kind, source) in enumerate(CIPHERTEXTS, 1):
    found = find_candidates(decode_source(kind, source), KEYS)
    print(f'Bản mã {i} ({kind}): {len(found)} ứng viên')
    for item in found:
        print(f'  Khóa {item["key"]}; {item["byteorder"]}; '
              f'{item["bytes"]} byte; {item["blocks"]} khối')
        print('  Bản rõ:', item['plaintext'])
        print('  Độ dài các khối rõ:', item['plain_lengths'])
    SOLUTIONS.append(found)

Bản mã 1 (Base64): 1 ứng viên
  Khóa 2; little; 63 byte; 3 khối
  Bản rõ: Ron Rivest, Adi Shamir and Leonard Adleman
  Độ dài các khối rõ: [20, 20, 2]
Bản mã 2 (Hex): 1 ứng viên
  Khóa 2; little; 20 byte; 1 khối
  Bản rõ: Meet-in-the-middle
  Độ dài các khối rõ: [18]
Bản mã 3 (Base64): 1 ứng viên
  Khóa 1; little; 46 byte; 46 khối
  Bản rõ: Vietnam National University - Ho Chi Minh City
  Độ dài các khối rõ: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Bản mã 4 (Nhị phân): 1 ứng viên
  Khóa 2; little; 21 byte; 1 khối
  Bản rõ: Man-in-the-middle
  Độ dài các khối rõ: [17]


## Sinh khóa ngẫu nhiên và thử thông điệp
Lời gọi không truyền p, q, e tự sinh đủ ba tham số. Khóa và số liệu thời gian thay đổi giữa
các lần chạy; kiểm tra kích thước n và hai chiều khôi phục. Không in bí mật của khóa sinh mới.

In [7]:
import platform
import time

started = time.perf_counter()
RANDOM_KEY = generate_key()
RANDOM_SECONDS = time.perf_counter() - started
random_packet = encrypt_text('Thực hành RSA – Nhóm 16\x00', RANDOM_KEY)
assert decrypt_text(random_packet, RANDOM_KEY) == 'Thực hành RSA – Nhóm 16\x00'
assert rsa_apply(rsa_apply(123456789, RANDOM_KEY.e, RANDOM_KEY.n),
                 RANDOM_KEY.d, RANDOM_KEY.n) == 123456789
print('Python:', platform.python_version())
print('Khóa sinh mới:', RANDOM_KEY.n.bit_length(), 'bit')
print('p khác q:', RANDOM_KEY.p != RANDOM_KEY.q)
print('e*d mod phi =', RANDOM_KEY.e * RANDOM_KEY.d % RANDOM_KEY.phi)
print('Khôi phục số và chuỗi UTF-8 có byte 0: ĐẠT')
print(f'Thời gian sinh khóa trong lần chạy này: {RANDOM_SECONDS:.3f} giây')

Python: 3.12.14
Khóa sinh mới: 2048 bit
p khác q: True
e*d mod phi = 1
Khôi phục số và chuỗi UTF-8 có byte 0: ĐẠT
Thời gian sinh khóa trong lần chạy này: 3.352 giây


## Nhập liệu tùy chọn
Đặt `RSA_INTERACTIVE = True` rồi chạy riêng cell. Bỏ trống cả p, q, e để sinh khóa 2048 bit;
cũng có thể bỏ trống một số tham số. Dùng tiền tố `0x` cho Hex, số còn lại đọc thập phân.
Chọn số hoặc chuỗi; số phải thuộc [0, n). Mặc định tắt để Run All không bị chờ.

In [8]:
RSA_INTERACTIVE = False

def read_optional_integer(prompt):
    text = input(prompt).strip()
    return None if not text else int(text, 16 if text.lower().startswith('0x') else 10)

if RSA_INTERACTIVE:
    try:
        user_p = read_optional_integer('p (bỏ trống để tự sinh): ')
        user_q = read_optional_integer('q (bỏ trống để tự sinh): ')
        user_e = read_optional_integer('e (bỏ trống để tự sinh): ')
        user_key = generate_key(user_p, user_q, user_e)
        print('PU =', user_key.public)
        mode = input('Nhập 1 cho số, 2 cho chuỗi: ').strip()
        if mode == '1':
            value = read_optional_integer('Số M: ')
            c = rsa_apply(value, user_key.e, user_key.n)
            print('Bản mã:', c)
            print('Khôi phục:', rsa_apply(c, user_key.d, user_key.n))
        elif mode == '2':
            user_packet = encrypt_text(input('Chuỗi: '), user_key)
            print('Gói dữ liệu:', user_packet)
            print('Khôi phục:', decrypt_text(user_packet, user_key))
        else:
            raise ValueError('Chỉ chọn 1 hoặc 2.')
    except (ValueError, TypeError) as error:
        print('Lỗi:', error)
else:
    print('Nhập liệu đang tắt. Đặt RSA_INTERACTIVE = True để dùng.')

Nhập liệu đang tắt. Đặt RSA_INTERACTIVE = True để dùng.


## Kiểm thử
Kiểm tra các giá trị độc lập của ba khóa và M = 5; toàn bộ 187 phần tử của khóa 1 ở cả
hai chiều; chuỗi nhiều khối, UTF-8, byte 0, dữ liệu rỗng, khóa thiếu tham số, bốn bản mã đề;
từ chối đầu vào ngoài miền và dữ liệu sai định dạng. Lỗi kiểm thử làm dừng cell.

In [9]:
import copy
import io
import unittest


class TestRSA(unittest.TestCase):
    def test_inverse_and_invalid_inverse(self):
        self.assertEqual(inverse_mod(7, 160), 23)
        self.assertEqual(inverse_mod(17, 3120), 2753)
        with self.assertRaises(ValueError):
            inverse_mod(6, 12)

    def test_primes_and_composites(self):
        for p in (2, 3, 11, 17, 65537):
            self.assertTrue(is_probable_prime(p))
        for c in (-1, 0, 1, 4, 9, 561, 1105, 1729, 3215031751):
            self.assertFalse(is_probable_prime(c))

    def test_known_three_keys(self):
        expected = [
            (187, 160, 7, 23),
            (13588476140342208394395166469647627226674348541791,
             13588476140342208394395145712936608632615990219784, 17,
             7993221259024828467291262184080358019185876599873),
            (101776877529005912638346811918779931246783058062684819617574643018368103302097,
             101776877529005912638346811918779931246144673982897083685478381401683682492836,
             886979, 24212225287904763939160097464943268930139828978795606022583874367720623008491),
        ]
        self.assertEqual([(k.n, k.phi, k.e, k.d) for k in KEYS], expected)
        for k in KEYS:
            self.assertEqual(k.d, pow(k.e, -1, k.phi))

    def test_m5_both_directions(self):
        self.assertEqual(rsa_apply(5, 7, 187), 146)
        self.assertEqual(rsa_apply(146, 23, 187), 5)
        self.assertEqual(rsa_apply(5, 23, 187), 180)
        self.assertEqual(rsa_apply(180, 7, 187), 5)

    def test_all_residues_small_key(self):
        for m in range(187):
            self.assertEqual(rsa_apply(rsa_apply(m, 7, 187), 23, 187), m)
            self.assertEqual(rsa_apply(rsa_apply(m, 23, 187), 7, 187), m)

    def test_invalid_keys(self):
        for args in [(11, 11, 7), (15, 17, 7), (11, 17, 8), (11, 17, 160),
                     (11, 17, 5), (True, 17, 7)]:
            with self.subTest(args=args), self.assertRaises(ValueError):
                generate_key(*args)

    def test_invalid_integer_messages(self):
        for m in (-1, 187, 188, 1.5, True, '5'):
            with self.subTest(m=m), self.assertRaises(ValueError):
                rsa_apply(m, 7, 187)

    def test_required_message_all_keys(self):
        for k in KEYS:
            self.assertEqual(decrypt_text(encrypt_text(MESSAGE, k), k), MESSAGE)

    def test_unicode_zero_and_boundaries(self):
        for k in KEYS[1:]:
            b, _ = block_sizes(k.n)
            for text in ('', '\x00', 'a\x00b\x00', 'Đại học Công nghệ Thông tin',
                         'A' * (b - 1), 'A' * b, 'A' * (b + 1), 'A' * (3 * b)):
                with self.subTest(bits=k.n.bit_length(), text=repr(text)):
                    self.assertEqual(decrypt_text(encrypt_text(text, k), k), text)

    def test_small_key_byte_limit(self):
        with self.assertRaises(ValueError):
            encrypt_text('ệ', KEYS[0])

    def test_fixed_cipher_width(self):
        for k in KEYS:
            b, width = block_sizes(k.n)
            packet = encrypt_text(MESSAGE, k)
            self.assertEqual(len(base64.b64decode(packet['base64'])),
                             ((len(MESSAGE) + b - 1) // b) * width)

    def test_four_given_plaintexts(self):
        self.assertEqual([[(x['key'], x['plaintext']) for x in found] for found in SOLUTIONS], [
            [(2, 'Ron Rivest, Adi Shamir and Leonard Adleman')],
            [(2, 'Meet-in-the-middle')],
            [(1, 'Vietnam National University - Ho Chi Minh City')],
            [(2, 'Man-in-the-middle')],
        ])

    def test_binary_bytes_and_endianness(self):
        raw = decode_source(*CIPHERTEXTS[3])
        self.assertEqual(raw.hex(), '2814ffb72ecaec67bf3f68cf309454f54ceeef5b04')
        self.assertEqual(SOLUTIONS[3][0]['byteorder'], 'little')

    def test_invalid_encodings_and_packet(self):
        for kind, value in [('Base64', '!'), ('Hex', 'ZZ'), ('Nhị phân', '01012'),
                            ('Nhị phân', '101')]:
            with self.subTest(kind=kind), self.assertRaises(ValueError):
                decode_source(kind, value)
        packet = copy.deepcopy(PACKETS[1])
        packet['base64'] = packet['base64'][:-4]
        with self.assertRaises(ValueError):
            decrypt_text(packet, KEYS[1])

    def test_missing_parameters(self):
        for kwargs in ({'p': 11, 'q': 17}, {'p': 11, 'e': 7, 'bits': 32},
                       {'q': 17, 'e': 7, 'bits': 32}, {'e': 65537, 'bits': 64}):
            k = generate_key(**kwargs)
            self.assertNotEqual(k.p, k.q)
            self.assertEqual(k.e * k.d % k.phi, 1)
            self.assertEqual(pow(pow(5, k.e, k.n), k.d, k.n), 5)

    def test_random_2048_bit_key(self):
        self.assertEqual(RANDOM_KEY.n.bit_length(), 2048)
        self.assertEqual(math.gcd(RANDOM_KEY.e, RANDOM_KEY.phi), 1)
        for m in (0, 1, 123456789, RANDOM_KEY.p, RANDOM_KEY.q, RANDOM_KEY.n - 1):
            self.assertEqual(pow(pow(m, RANDOM_KEY.e, RANDOM_KEY.n), RANDOM_KEY.d, RANDOM_KEY.n), m)


test_log = io.StringIO()
TEST_RESULT = unittest.TextTestRunner(stream=test_log, verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(TestRSA))
print(test_log.getvalue())
assert TEST_RESULT.wasSuccessful(), 'Có kiểm thử thất bại.'

test_all_residues_small_key (__main__.TestRSA.test_all_residues_small_key) ... ok
test_binary_bytes_and_endianness (__main__.TestRSA.test_binary_bytes_and_endianness) ... ok
test_fixed_cipher_width (__main__.TestRSA.test_fixed_cipher_width) ... ok
test_four_given_plaintexts (__main__.TestRSA.test_four_given_plaintexts) ... ok
test_invalid_encodings_and_packet (__main__.TestRSA.test_invalid_encodings_and_packet) ... ok
test_invalid_integer_messages (__main__.TestRSA.test_invalid_integer_messages) ... ok
test_invalid_keys (__main__.TestRSA.test_invalid_keys) ... ok
test_inverse_and_invalid_inverse (__main__.TestRSA.test_inverse_and_invalid_inverse) ... ok
test_known_three_keys (__main__.TestRSA.test_known_three_keys) ... ok
test_m5_both_directions (__main__.TestRSA.test_m5_both_directions) ... ok
test_missing_parameters (__main__.TestRSA.test_missing_parameters) ... ok
test_primes_and_composites (__main__.TestRSA.test_primes_and_composites) ... ok
test_random_2048_bit_key (__main__.TestR

## Nhận xét
- Khóa 1 có 8 bit nên chỉ minh họa; khóa 2 có 164 bit, khóa 3 có 256 bit, đều quá nhỏ cho sử dụng thực tế.
- Bản rõ yêu cầu có 40 byte UTF-8. Khóa 1, 2, 3 lần lượt tạo 40, 2, 2 khối; biểu diễn bản mã có 40, 42, 64 byte.
- Cùng khóa, cùng quy ước và cùng bản rõ cho cùng bản mã do không có padding ngẫu nhiên.
  Đổi thứ tự byte hoặc quy ước chia khối sẽ đổi Base64 dù toán học RSA không đổi.
- Khóa lớn hơn không khắc phục thiếu padding, thiếu xác thực hay lỗi quản lý khóa.
  Chương trình dùng để quan sát RSA; thực tế dùng thư viện đã kiểm định với RSA-OAEP/RSA-PSS [2].
- Các bản mã đề được khôi phục bằng khóa 2, 2, 1, 2. Không bản mã nào trong bốn mẫu cần khóa 3.
  Bản mã Hex số 2 chỉ có 20 byte, ngắn hơn độ rộng cố định 21 byte của khóa 2;
  đọc little-endian tương đương bổ sung byte 00 ở cuối. Việc giải mã mẫu không có siêu dữ liệu
  không bảo đảm khôi phục byte 0 ở cuối một thông điệp tùy ý.

## Tài liệu tham khảo
[1] Đề NT101 Lab03, Nhiệm vụ 2.1, trang PDF 3–7, tệp được cung cấp trong học phần.  
[2] RFC 8017, mục 4, 5, 7, 8: https://www.rfc-editor.org/rfc/rfc8017.html.  
[3] Python, `pow`: https://docs.python.org/3/library/functions.html#pow.  
[4] Python, `secrets`: https://docs.python.org/3/library/secrets.html.